# Recorrido del código, paso a paso

Este notebook pasa **un ejemplo pequeño** por cada función del proyecto, en el mismo orden en que se
ejecutan, y muestra la tabla antes y después de cada paso. La idea es que cada línea de código se
entienda viendo lo que hace con datos concretos.

**El ejemplo:** 7 intentos de verificación de 4 cuentas, con nombres legibles:

- `cuenta-1` intenta 4 veces en pocos días y la rechazan por foto falsa y fotocopia.
- `cuenta-2` usa **el mismo documento** que `cuenta-1` (`doc-1`).
- `cuenta-3` es un cliente nuevo con señales limpias.
- `cuenta-4` es del cliente A, que nombra distinto las columnas (`Distrust` en minúscula).

> Requisito: haber ejecutado `uv run python -m src.pipeline` (el paso 6 usa el modelo entrenado).

In [1]:
import os
os.chdir("..")  # el proyecto se ejecuta desde la raíz, donde está config.yaml

import numpy as np
import pandas as pd

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 30)

## Paso 1. `src/config.py`: la configuración

`load_config` lee `config.yaml` y lo devuelve como un diccionario. Todo el código toma de aquí los
nombres de columnas, las variables, los costos…

In [2]:
from src.config import load_config

cfg = load_config()
print("columns:", cfg["columns"])
print("unify:  ", cfg["unify"])
print("costos: ", cfg["decision"]["costs"])

columns: {'id': 'PROCESS_ID', 'account': 'ACCOUNT_ID', 'date': 'DATE', 'target': 'IS_FRAUD'}
unify:   {'DISTRUST_U': ['DISTRUST', 'Distrust'], 'ANOMALY_U': ['ANOMALY', 'Anomaly'], 'LIFETIME_U': ['LIFETIME', 'Lifetime']}
costos:  {'fraude_aprobado': 1.0, 'legitimo_rechazado': 2.0}


## Paso 2. El ejemplo, en el formato del CSV original

Cada fila es un intento de verificación. Las columnas son las mismas que trae `data_anonymized.csv`
(solo las que usa el modelo).

In [3]:
CLIENTE = {nombre: cid for cid, nombre in cfg["clients"].items()}   # "A"/"B" -> hash del cliente

filas = [
    # proceso, cuenta,    fecha,              cliente, DISTRUST, Distrust, LIFETIME, Lifetime, documento, email,  teléfono, STATUS,    DECLINED_REASON,           IS_FRAUD
    ("p1", "cuenta-1", "2024-08-01T10:00:00-05:00", "B", 0.90, None, 30,   None, "doc-1", "e-1", "t-1", "failure", "portrait_photo_is_fake",  1),
    ("p2", "cuenta-1", "2024-08-01T15:00:00-05:00", "B", 0.90, None, 30,   None, "doc-1", "e-1", "t-1", "failure", "document_is_a_photocopy", 1),
    ("p3", "cuenta-1", "2024-08-02T09:00:00-05:00", "B", 0.90, None, 31,   None, "doc-1", "e-1", "t-1", "success", None,                      1),
    ("p4", "cuenta-2", "2024-08-03T12:00:00-05:00", "B", 0.50, None, 200,  None, "doc-1", "e-2", "t-2", "success", None,                      1),
    ("p5", "cuenta-3", "2024-08-04T08:00:00-05:00", "B", 0.00, None, 2400, None, "doc-3", "e-3", "t-3", "success", None,                      0),
    ("p6", "cuenta-4", "2024-08-04T09:00:00-05:00", "A", None, 0.20, None, 500,  "doc-4", "e-4", None,  "success", None,                      0),
    ("p7", "cuenta-1", "2024-08-05T09:00:00-05:00", "B", 0.90, None, 34,   None, "doc-1", "e-1", "t-1", None,      None,                      1),
]
crudo = pd.DataFrame(filas, columns=[
    "PROCESS_ID", "ACCOUNT_ID", "DATE", "CLIENTE_", "DISTRUST", "Distrust", "LIFETIME", "Lifetime",
    "DOCUMENT_NUMBER", "EMAIL", "PHONE_NUMBER", "STATUS", "DECLINED_REASON", "IS_FRAUD"])
crudo["CLIENT_ID"] = crudo.pop("CLIENTE_").map(CLIENTE)
crudo["ANOMALY"] = np.where(crudo["CLIENT_ID"] == CLIENTE["B"], 0.17, np.nan)
crudo["Anomaly"] = np.where(crudo["CLIENT_ID"] == CLIENTE["A"], 0.17, np.nan)
crudo["DOCUMENT_TYPE"] = np.where(crudo["CLIENT_ID"] == CLIENTE["B"], "national-id", None)
crudo.drop(columns="CLIENT_ID")

,PROCESS_ID,ACCOUNT_ID,DATE,DISTRUST,Distrust,LIFETIME,Lifetime,DOCUMENT_NUMBER,EMAIL,PHONE_NUMBER,STATUS,DECLINED_REASON,IS_FRAUD,ANOMALY,Anomaly,DOCUMENT_TYPE
0,p1,cuenta-1,2024-08-01T10:00:00-05:00,0.9,NaN,30.0,NaN,doc-1,e-1,t-1,failure,portrait_photo_is_fake,1,0.17,NaN,national-id
1,p2,cuenta-1,2024-08-01T15:00:00-05:00,0.9,NaN,30.0,NaN,doc-1,e-1,t-1,failure,document_is_a_photocopy,1,0.17,NaN,national-id
2,p3,cuenta-1,2024-08-02T09:00:00-05:00,0.9,NaN,31.0,NaN,doc-1,e-1,t-1,success,NaN,1,0.17,NaN,national-id
3,p4,cuenta-2,2024-08-03T12:00:00-05:00,0.5,NaN,200.0,NaN,doc-1,e-2,t-2,success,NaN,1,0.17,NaN,national-id
4,p5,cuenta-3,2024-08-04T08:00:00-05:00,0.0,NaN,2400.0,NaN,doc-3,e-3,t-3,success,NaN,0,0.17,NaN,national-id
5,p6,cuenta-4,2024-08-04T09:00:00-05:00,NaN,0.2,NaN,500.0,doc-4,e-4,NaN,success,NaN,0,NaN,0.17,NaN
6,p7,cuenta-1,2024-08-05T09:00:00-05:00,0.9,NaN,34.0,NaN,doc-1,e-1,t-1,NaN,NaN,1,0.17,NaN,national-id


Fíjate en `p7`: es el intento **actual**, el que hay que decidir. Su `STATUS` está vacío porque el
proceso todavía no terminó.

## Paso 3. `src/data.py` → `prepare_raw`: validar y tipar

Comprueba que estén las columnas obligatorias, que no haya procesos repetidos ni fechas vacías, y
convierte la fecha de texto a fecha real (en UTC).

In [4]:
from src.data import DataValidationError, prepare_raw

print("Antes:  ", crudo["DATE"].dtype, "->", crudo["DATE"].iloc[0])
datos = prepare_raw(crudo, cfg["columns"])
print("Después:", datos["DATE"].dtype, "->", datos["DATE"].iloc[0])

Antes:   str -> 2024-08-01T10:00:00-05:00
Después: datetime64[us, UTC] -> 2024-08-01 15:00:00+00:00


Si los datos están mal, falla con un mensaje claro (la API lo convierte en un error 422):

In [5]:
malo = crudo.copy()
malo.loc[0, "DATE"] = "ayer por la tarde"
try:
    prepare_raw(malo, cfg["columns"])
except DataValidationError as e:
    print("DataValidationError:", str(e)[:80], "...")

DataValidationError: Fecha inválida en DATE: Time data ayer por la tarde is not ISO8601 format. You m ...


## Paso 4. `src/data.py` → `clean`: unificar los dos esquemas

Hallazgo 1 del EDA: el cliente A escribe `Distrust` y el B `DISTRUST`; nunca están llenas las dos.
`clean` las junta en `DISTRUST_U` con `fillna`: *"toma `DISTRUST`; si está vacía, usa `Distrust`"*.

In [6]:
from src.data import clean

limpio = clean(datos, cfg)
limpio[["PROCESS_ID", "DISTRUST", "Distrust", "DISTRUST_U", "LIFETIME", "Lifetime", "LIFETIME_U",
        "CLIENTE", "DOCUMENT_TYPE"]]

,PROCESS_ID,DISTRUST,Distrust,DISTRUST_U,LIFETIME,Lifetime,LIFETIME_U,CLIENTE,DOCUMENT_TYPE
0,p1,0.9,NaN,0.9,30.0,NaN,30.0,B,national-id
1,p2,0.9,NaN,0.9,30.0,NaN,30.0,B,national-id
2,p3,0.9,NaN,0.9,31.0,NaN,31.0,B,national-id
3,p4,0.5,NaN,0.5,200.0,NaN,200.0,B,national-id
4,p5,0.0,NaN,0.0,2400.0,NaN,2400.0,B,national-id
5,p6,NaN,0.2,0.2,NaN,500.0,500.0,A,desconocido
6,p7,0.9,NaN,0.9,34.0,NaN,34.0,B,national-id


`p6` (cliente A) tenía `DISTRUST` vacío y `Distrust = 0.20`: ahora `DISTRUST_U = 0.20`. Además, el
hash del cliente se tradujo a `A`/`B`, y el tipo de documento vacío pasó a `"desconocido"`.

## Paso 5. `src/features.py` → `build_features`: variables que solo miran el pasado

Primero se ordena por fecha. Después, cada variable usa **operaciones acumuladas**, que por definición
solo ven las filas anteriores. Las armo a mano, una por una, para ver el truco.

In [7]:
df = limpio.sort_values(["DATE", "PROCESS_ID"]).reset_index(drop=True)
fallo = (df["STATUS"] == "failure").astype(int)      # 1 si ese intento falló; el actual (vacío) da 0
cuenta = df.groupby("ACCOUNT_ID")

paso = df[["PROCESS_ID", "ACCOUNT_ID", "STATUS"]].copy()
paso["fallo"] = fallo
paso["intentos_previos"] = cuenta.cumcount()                       # 0, 1, 2… dentro de cada cuenta
paso["cumsum_fallo"] = fallo.groupby(df["ACCOUNT_ID"]).cumsum()     # incluye la fila actual
paso["fallos_previos"] = paso["cumsum_fallo"] - paso["fallo"]       # al restarla, queda solo el pasado
paso

,PROCESS_ID,ACCOUNT_ID,STATUS,fallo,intentos_previos,cumsum_fallo,fallos_previos
0,p1,cuenta-1,failure,1,0,1,0
1,p2,cuenta-1,failure,1,1,2,1
2,p3,cuenta-1,success,0,2,2,2
3,p4,cuenta-2,success,0,0,0,0
4,p5,cuenta-3,success,0,0,0,0
5,p6,cuenta-4,success,0,0,0,0
6,p7,cuenta-1,NaN,0,3,2,2


Lee la `cuenta-1` de arriba abajo:

- `p1`: es su primer intento → `intentos_previos = 0`, `fallos_previos = 0`.
- `p2`: tiene 1 intento antes, que falló → `1` y `1`.
- `p3`: 2 intentos antes, los dos fallidos → `2` y `2`.
- `p7` (el actual): 3 intentos antes, 2 fallidos → `3` y `2`.

**Por qué `cumsum − fallo`:** la suma acumulada de `p2` es 2 porque incluye su propio fallo. Ese
resultado no se conoce al decidir, así que se resta: queda 1, que es lo que había *antes*.

**La velocidad** (`intentos_24h`): cuántos intentos tuvo la cuenta en las 24 horas anteriores.

In [8]:
segundos = (df["DATE"] - pd.Timestamp("1970-01-01", tz="UTC")).dt.total_seconds()
t = segundos[df["ACCOUNT_ID"] == "cuenta-1"].to_numpy()          # tiempos de la cuenta-1, en orden
desde = np.searchsorted(t, t - 24 * 3600, side="left")             # dónde empieza la ventana de cada uno
pd.DataFrame({"proceso": df.loc[df["ACCOUNT_ID"] == "cuenta-1", "PROCESS_ID"].to_numpy(),
              "horas_desde_el_primero": (t - t[0]) / 3600,
              "posición": np.arange(len(t)),
              "inicio_de_la_ventana": desde,
              "intentos_24h": np.arange(len(t)) - desde})

,proceso,horas_desde_el_primero,posición,inicio_de_la_ventana,intentos_24h
0,p1,0.0,0,0,0
1,p2,5.0,1,0,1
2,p3,23.0,2,0,2
3,p7,95.0,3,3,0


`p2` ocurrió 5 horas después de `p1`, así que `p1` está dentro de su ventana → 1. `p7` ocurrió días
después: no hay intentos en las 24 horas anteriores → 0.

**La identidad compartida** (`cuentas_previas_documento`): cuántas *otras* cuentas usaron antes el
mismo documento.

In [9]:
paso = df[["PROCESS_ID", "ACCOUNT_ID", "DOCUMENT_NUMBER"]].copy()
paso["pareja_nueva"] = (~df.duplicated(["DOCUMENT_NUMBER", "ACCOUNT_ID"])).astype(int)  # 1ª vez de (doc, cuenta)
paso["cuentas_vistas"] = paso["pareja_nueva"].groupby(df["DOCUMENT_NUMBER"]).cumsum()  # cuentas distintas hasta aquí
paso["cuentas_previas_documento"] = paso["cuentas_vistas"] - 1                          # menos la cuenta actual
paso

,PROCESS_ID,ACCOUNT_ID,DOCUMENT_NUMBER,pareja_nueva,cuentas_vistas,cuentas_previas_documento
0,p1,cuenta-1,doc-1,1,1,0
1,p2,cuenta-1,doc-1,0,1,0
2,p3,cuenta-1,doc-1,0,1,0
3,p4,cuenta-2,doc-1,1,2,1
4,p5,cuenta-3,doc-3,1,1,0
5,p6,cuenta-4,doc-4,1,1,0
6,p7,cuenta-1,doc-1,0,2,1


`doc-1` lo usa primero la `cuenta-1`. Cuando aparece la `cuenta-2` (`p4`) con el mismo documento,
ya hay otra cuenta antes → 1. En el EDA, un documento compartido entre cuentas tiene 45 % de fraude.

Ahora la función real, que calcula todo eso (y más) de una vez:

In [10]:
from src.features import build_features

variables = build_features(limpio, cfg)
variables[["PROCESS_ID", "ACCOUNT_ID", "intentos_previos", "fallos_previos", "rechazos_riesgo_previos",
           "intentos_24h", "horas_desde_intento_anterior", "cuentas_previas_documento", "falta_telefono"]]

,PROCESS_ID,ACCOUNT_ID,intentos_previos,fallos_previos,rechazos_riesgo_previos,intentos_24h,horas_desde_intento_anterior,cuentas_previas_documento,falta_telefono
0,p1,cuenta-1,0,0,0,0,NaN,0,0
1,p2,cuenta-1,1,1,1,1,5.0,0,0
2,p3,cuenta-1,2,2,2,2,18.0,0,0
3,p4,cuenta-2,0,0,0,0,NaN,1,0
4,p5,cuenta-3,0,0,0,0,NaN,0,0
5,p6,cuenta-4,0,0,0,0,NaN,0,1
6,p7,cuenta-1,3,2,2,0,72.0,1,0


`rechazos_riesgo_previos` de `p7` es 2: los rechazos de `p1` (foto falsa) y `p2` (fotocopia) están
en la lista `risk_reasons` del config. `falta_telefono = 1` en `p6`, que no tiene teléfono.

## Paso 6. `src/models.py`: el modelo

El modelo entrenado es un `Pipeline`: primero el preprocesamiento, después LightGBM. Recibe las 22
variables y devuelve la probabilidad de fraude.

In [11]:
import joblib

artefacto = joblib.load("models/modelo.joblib")
modelo = artefacto["pipeline"]
X = variables[artefacto["numeric"] + artefacto["categorical"]]
print(modelo)
print("Variables que recibe:", len(X.columns))

Pipeline(steps=[('pre',
                 ColumnTransformer(transformers=[('num', 'passthrough',
                                                  ['DISTRUST_U', 'ANOMALY_U',
                                                   'LIFETIME_U',
                                                   'intentos_previos',
                                                   'fallos_previos',
                                                   'tasa_fallos_previos',
                                                   'rechazos_riesgo_previos',
                                                   'horas_desde_intento_anterior',
                                                   'intentos_24h',
                                                   'cuentas_previas_documento',
                                                   'cuentas_previas_email',
                                                   'cuentas_previas_telefono',
                                                   'intentos_previos_document...
   

In [12]:
variables["score"] = modelo.predict_proba(X)[:, 1]   # columna 1 = probabilidad de fraude
variables[["PROCESS_ID", "ACCOUNT_ID", "DISTRUST_U", "LIFETIME_U", "rechazos_riesgo_previos", "score"]].round(3)

,PROCESS_ID,ACCOUNT_ID,DISTRUST_U,LIFETIME_U,rechazos_riesgo_previos,score
0,p1,cuenta-1,0.9,30.0,0,0.473
1,p2,cuenta-1,0.9,30.0,1,0.729
2,p3,cuenta-1,0.9,31.0,2,0.788
3,p4,cuenta-2,0.5,200.0,0,0.486
4,p5,cuenta-3,0.0,2400.0,0,0.057
5,p6,cuenta-4,0.2,500.0,0,0.048
6,p7,cuenta-1,0.9,34.0,2,0.942


La `cuenta-3` (sin desconfianza, teléfono de 2.400 días) recibe un score bajo; el intento actual de la
`cuenta-1` (`p7`), uno muy alto.

**Monotonía:** subir `DISTRUST` nunca puede bajar el score. Lo compruebo con `p5`:

In [13]:
fila = X.iloc[[variables.index[variables["PROCESS_ID"] == "p5"][0]]]
prueba = pd.concat([fila] * 6, ignore_index=True)
prueba["DISTRUST_U"] = [0.0, 0.2, 0.4, 0.6, 0.8, 1.0]
pd.DataFrame({"DISTRUST_U": prueba["DISTRUST_U"], "score": modelo.predict_proba(prueba)[:, 1].round(3)})

,DISTRUST_U,score
0,0.0,0.057
1,0.2,0.080
2,0.4,0.096
3,0.6,0.126
4,0.8,0.244
5,1.0,0.263


## Paso 7. `src/decision.py`: de probabilidad a decisión

Para cada proceso se calcula cuánto costaría, en promedio, cada acción, y se elige la más barata.
Con probabilidad de fraude `p` (y un fraude aprobado = 1):

- **aprobar** cuesta `p × 1` (si es fraude, lo dejamos pasar)
- **rechazar** cuesta `(1 − p) × 2` (si es bueno, lo perdimos; supuesto S6: vale el doble)
- **verificar** cuesta `0,05 + (1 − p) × 0,10 × 2 + p × 0,30 × 1` (el costo de verificar, más el 10 %
  de buenos que abandona, más el 30 % de fraudes que la pasan)

In [14]:
dcfg = cfg["decision"]
su = dcfg["step_up"]
filas = []
for p in [0.05, 0.20, 0.30, 0.50, 0.80, 0.90]:
    aprobar = p * 1
    rechazar = (1 - p) * 2
    verificar = su["costo"] + (1 - p) * su["abandono_legitimos"] * 2 + p * (1 - su["captura_fraude"]) * 1
    costos = {"aprobar": aprobar, "verificar": verificar, "rechazar": rechazar}
    filas.append({"p": p, **{k: round(v, 3) for k, v in costos.items()}, "más barata": min(costos, key=costos.get)})
pd.DataFrame(filas)

,p,aprobar,verificar,rechazar,más barata
0,0.05,0.05,0.255,1.9,aprobar
1,0.20,0.20,0.270,1.6,aprobar
2,0.30,0.30,0.280,1.4,verificar
3,0.50,0.50,0.300,1.0,verificar
4,0.80,0.80,0.330,0.4,verificar
5,0.90,0.90,0.340,0.2,rechazar


Los umbrales son exactamente los puntos donde cambia la acción más barata. `cost_thresholds` los
calcula con una fórmula (despejando `p` en "costo de A = costo de B"):

In [15]:
from src.decision import cost_thresholds, decide

umbrales = cost_thresholds(dcfg)
print({k: round(v, 3) for k, v in umbrales.items()})
variables["decision"] = decide(variables["score"], umbrales)
variables[["PROCESS_ID", "score", "decision"]].round(3)

{'verificar': 0.278, 'rechazar': 0.833}


,PROCESS_ID,score,decision
0,p1,0.473,verificar
1,p2,0.729,verificar
2,p3,0.788,verificar
3,p4,0.486,verificar
4,p5,0.057,aprobar
5,p6,0.048,aprobar
6,p7,0.942,rechazar


## Paso 8. `src/inference.py` → `contributions` y `reasons`: explicar cada decisión

LightGBM descompone cada score en **cuánto aportó cada variable** (valores SHAP). Son exactos: el
valor base más la suma de los aportes da el score del modelo (en log-odds).

In [16]:
from src.inference import contributions, reasons

contrib, base = contributions(modelo, X)
i = variables.index[variables["PROCESS_ID"] == "p7"][0]
aportes = contrib.iloc[i].sort_values(ascending=False)
print("Aportes de p7 (los 5 mayores):")
print(aportes.head(5).round(3).to_string())

Aportes de p7 (los 5 mayores):
DISTRUST_U                   1.180
rechazos_riesgo_previos      1.009
LIFETIME_U                   0.740
cuentas_previas_documento    0.520
intentos_previos_email       0.430


In [17]:
logodds = base[i] + contrib.iloc[i].sum()
print(f"valor base {base[i]:.3f} + suma de aportes {contrib.iloc[i].sum():.3f} = {logodds:.3f} (log-odds)")
print(f"convertido a probabilidad: {1 / (1 + np.exp(-logodds)):.4f}")
print(f"score del modelo:          {variables.loc[i, 'score']:.4f}")

valor base -2.122 + suma de aportes 4.913 = 2.791 (log-odds)
convertido a probabilidad: 0.9422
score del modelo:          0.9422


`reasons` toma los 3 mayores aportes y los escribe en lenguaje legible. Las aprobaciones no llevan
motivos:

In [18]:
variables["motivos"] = reasons(contrib, X, variables["decision"], cfg["explanations"])
for _, r in variables.iterrows():
    print(f"{r.PROCESS_ID}  {r.decision:9}  score {r.score:.2f}  ", [m["descripcion"] for m in r.motivos])

p1  verificar  score 0.47   ['Desconfianza del proveedor de verificación: 0.90', 'Antigüedad del teléfono (días): 30', 'Tipo de documento: national-id']
p2  verificar  score 0.73   ['Rechazos previos por señales de fraude: 1', 'Desconfianza del proveedor de verificación: 0.90', 'Antigüedad del teléfono (días): 30']
p3  verificar  score 0.79   ['Rechazos previos por señales de fraude: 2', 'Desconfianza del proveedor de verificación: 0.90', 'Antigüedad del teléfono (días): 31']
p4  verificar  score 0.49   ['Otras cuentas que usaron el mismo documento: 1', 'Antigüedad del teléfono (días): 200', 'Fallos previos con el mismo documento: 2']
p5  aprobar    score 0.06   []
p6  aprobar    score 0.05   []
p7  rechazar   score 0.94   ['Desconfianza del proveedor de verificación: 0.90', 'Rechazos previos por señales de fraude: 2', 'Antigüedad del teléfono (días): 34']


## Paso 9. Todo junto: `Scorer`, lo que usan la CLI y la API

`Scorer.score` hace los pasos 3 a 8 en una llamada, sobre procesos nuevos en el formato crudo. Para
calcular el historial usa todo el dataset histórico además de lo que recibe.

In [19]:
from src.inference import Scorer

scorer = Scorer()
nuevos = crudo.drop(columns="IS_FRAUD")             # en producción no hay etiqueta
resultado, _ = scorer.score(nuevos)
resultado[["PROCESS_ID", "score", "decision"]].assign(
    motivos=[" | ".join(m["descripcion"] for m in ms) for ms in resultado["motivos"]])

,PROCESS_ID,score,decision,motivos
0,p1,0.4729,verificar,Desconfianza del proveedor de verificación: 0....
1,p2,0.7288,verificar,Rechazos previos por señales de fraude: 1 | De...
2,p3,0.7885,verificar,Rechazos previos por señales de fraude: 2 | De...
3,p4,0.4857,verificar,Otras cuentas que usaron el mismo documento: 1...
4,p5,0.0574,aprobar,
5,p6,0.0478,aprobar,
6,p7,0.9422,rechazar,Desconfianza del proveedor de verificación: 0....


## Paso 10. `src/monitor.py` → `psi`: detectar deriva

El PSI compara la distribución de una variable en producción con la de entrenamiento. Cerca de 0 es
estable; más de 0,25 es deriva.

In [20]:
from src.monitor import psi

rng = np.random.default_rng(0)
entrenamiento = pd.Series(rng.beta(2, 5, 5000))           # como DISTRUST en train
igual = pd.Series(rng.beta(2, 5, 500))                    # un lote con la misma distribución
desplazado = (igual + 0.3).clip(0, 1)                     # un proveedor que recalibra su puntaje
print(f"PSI lote normal:     {psi(entrenamiento, igual):.3f}")
print(f"PSI lote desplazado: {psi(entrenamiento, desplazado):.3f}")

PSI lote normal:     0.024
PSI lote desplazado: 4.784


## Resumen

| Paso | Archivo y función | Qué hizo con el ejemplo |
|---|---|---|
| 3 | `data.prepare_raw` | Validó y convirtió la fecha |
| 4 | `data.clean` | Juntó `DISTRUST` y `Distrust`; tradujo el cliente |
| 5 | `features.build_features` | Calculó el historial de cada cuenta y documento, solo con el pasado |
| 6 | `models` (el Pipeline guardado) | Dio una probabilidad de fraude por proceso |
| 7 | `decision.cost_thresholds` y `decide` | Eligió la acción más barata |
| 8 | `inference.contributions` y `reasons` | Explicó cada verificación o rechazo |
| 9 | `inference.Scorer` | Hizo todo junto, como la API |
| 10 | `monitor.psi` | Midió si los datos cambiaron |